> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 02 — G/E, A+D and A×E covariance with analytical SE

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str);train.columns=train.columns.str.replace('regressedlr_','',regex=False)
store=vigor.open_store(ANALYSIS_STORE,mode='a',lazy=False)
Ga=vigor.open_grm(store,component='additive',materialize=True);Gd=vigor.open_grm(store,component='dominance',materialize=True)
fit_mask=train.notna().any(axis=1).to_numpy()
# Resolve environment.
pdr=pd.read_csv(PROCESSED_DATA_CSV,dtype={'rfid':str}).set_index('rfid').reindex(train.index)
if GXE_ENVIRONMENT_COLUMN is not None: env=pdr[GXE_ENVIRONMENT_COLUMN]
else:
    sex=[c for c in pdr.columns if str(c).lower()=='sex']; env=pdr[sex[0]] if sex else None
if env is None: raise ValueError('Set GXE_ENVIRONMENT_COLUMN; no sex column was found.')
kernels,gxe_info=vigor.make_gxe_kernels({'additive':Ga,'dominance':Gd},env,index=train.index,fit_mask=fit_mask,include_gxe=True,gxe_components=GXE_COMPONENTS,dtype=np.float64)

In [ ]:
common=dict(
    solver=FULL_SOLVER,
    variance_closure=VARIANCE_CLOSURE,
    variance_closure_strength=VARIANCE_CLOSURE_STRENGTH,
    analytic_se=ANALYTIC_SE,
    analytic_se_probes=ANALYTIC_PROBES,
    analytic_trace_probes=ANALYTIC_TRACE_PROBES,
    n_boot=0,random_state=SEED,blas_threads=BLAS_THREADS,progress=True,
)
fit_ge_low=vigor.genetic_varcov_PSD(
    Ga,train,rank_G=min(LOW_RANK_G,train.shape[1]),rank_E=min(LOW_RANK_E,train.shape[1]),**common)
fit_ge_full=vigor.genetic_varcov_PSD(
    Ga,train,rank_G=FULL_RANK_G,rank_E=FULL_RANK_E,**common)
fit_multi=vigor.genetic_varcov_PSD(
    kernels,train,rank_G=FULL_RANK_G,rank_E=FULL_RANK_E,**common)

# Persist the scientific point estimates/SEs, not Python optimizer objects.
store.write_covariance_result(fit_ge_low,trait_names=train.columns,group='covariance/GE_rank10',mode='a')
store.write_covariance_result(fit_ge_full,trait_names=train.columns,group='covariance/GE_full',mode='a')
store.write_covariance_result(fit_multi,trait_names=train.columns,group='covariance/ADGxE_full',mode='a')
print('Persisted covariance groups: GE_rank10, GE_full, ADGxE_full')


In [ ]:
rows=[]
for name,fit in [('GE_rank10',fit_ge_low),('GE_full',fit_ge_full),('ADGxE_full',fit_multi)]:
    org=vigor.organize_scores(train,fit)
    h=vigor.as_frame(org['heritability_table'])
    row={
        'model':name,
        'median_h2':float(np.nanmedian(h['h2'])),
        'median_h2_phenotype':float(np.nanmedian(h.get('h2_phenotype',np.nan))),
        'median_closure_ratio':float(np.nanmedian(h.get('variance_closure_ratio',np.nan))),
        'closure_flags':int(np.nansum(h.get('variance_closure_flag',False))),
        'finite_rG_SE_fraction':float(np.isfinite(vigor.as_frame(org['g_corr_table']).get('rG_SE',np.nan)).mean()),
    }
    rows.append(row)
    h.to_csv(RESULTS/f'{name}_heritability.csv')
summary=pd.DataFrame(rows)
summary.to_csv(RESULTS/'covariance_model_summary.csv',index=False)
display(summary)


Both rank-10 and full-rank models deliberately use **L-BFGS**. Differences therefore reflect rank/model structure rather than the spectral-solver pathology observed in the real data.